Follow along this tutorial https://github.com/alexeygrigorev/rag-agents-workshop

In [1]:
!pip install minsearch

In [2]:
import requests 

docs_url = 'https://github.com/alexeygrigorev/llm-rag-workshop/raw/main/notebooks/documents.json'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()

documents = []

for course in documents_raw:
    course_name = course['course']

    for doc in course['documents']:
        doc['course'] = course_name
        documents.append(doc)

In [3]:
documents[2]

{'text': "Yes, even if you don't register, you're still eligible to submit the homeworks.\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute.",
 'section': 'General course-related questions',
 'question': 'Course - Can I still join the course after the start date?',
 'course': 'data-engineering-zoomcamp'}

In [4]:
from minsearch import AppendableIndex

index = AppendableIndex(
    text_fields=["question", "text", "section"],
    keyword_fields=["course"]
)

index.fit(documents)

In [5]:
index.search('Can i still join the course?')

[{'text': 'Yes, you can. You won’t be able to submit some of the homeworks, but you can still take part in the course.\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers’ Projects by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.',
  'section': 'General course-related questions',
  'question': 'The course has already started. Can I still join it?',
  'course': 'machine-learning-zoomcamp'},
 {'text': "Yes, even if you don't register, you're still eligible to submit the homeworks.\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute.",
  'section': 'General course-related questions',
  'question': 'Course - Can I still join the course after the start date?',
  'course': 'data-engineering-zoomcamp'},
 {'text': "Here’s how you join a in Slack: https://slack.com/

In [6]:
def search(query):
    boost = {'question': 3.0, 'section': 0.5}

    results = index.search(
        query=query,
        filter_dict={'course': 'data-engineering-zoomcamp'},
        boost_dict=boost,
        num_results=5,
        output_ids=True
    )

    return results

In [8]:
question = "can i still join the course?"

In [9]:
prompt_template = """
You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.

<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(query, search_results):
    context = ""

    for doc in search_results:
        context = context + f"section: {doc['section']}\nquestion: {doc['question']}\nanswer: {doc['text']}\n\n"
    
    prompt = prompt_template.format(question=query, context=context).strip()
    return prompt

In [10]:
search_results = search(question)

In [19]:
prompt = build_prompt(question, search_results)

In [15]:
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
print(api_key[:4])

sk-p


In [17]:
from openai import OpenAI
client = OpenAI()

def llm(prompt):
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content


In [20]:
answer = llm(prompt)

In [21]:
print(answer)

Yes, you can still join the course after the start date. Even if you don't register, you're eligible to submit the homeworks. However, keep in mind that there will be deadlines for turning in the final projects, so it's best not to leave everything to the last minute.


In [22]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt)
    return answer

In [23]:
rag("How do i run in docker?")

'To run Docker, you can use the following command:\n\n```bash\ndocker run -it \\\n-e POSTGRES_USER="root" \\\n-e POSTGRES_PASSWORD="root" \\\n-e POSTGRES_DB="ny_taxi" \\\n-p 5432:5432 \\\npostgres:13\n```\n\nIf you\'re attempting to run this command a second time and encounter an error related to the mount source path, it may indicate that you should not mount the volume again. In that case, you can omit the `-v <your path>:/var/lib/postgresql/data` option in the second run. \n\nIf you\'re using Windows and encounter the error "the input device is not a TTY," try prefixing your command with `winpty` like this:\n\n```bash\nwinpty docker run -it ubuntu bash\n```\n\nAdditionally, it\'s recommended to run Docker commands from a Linux distribution file system in WSL (Windows Subsystem for Linux) if you\'re on Windows Home Edition.'

## "Agentic" RAG

In [24]:
prompt_template = """
You're a course teaching assistant.

You're given a QUESTION from a course student and that you need to answer with your own knowledge and provided CONTEXT.
At the beginning the context is EMPTY.

<QUESTION>
{question}
</QUESTION>

<CONTEXT> 
{context}
</CONTEXT>

If CONTEXT is EMPTY, you can use our FAQ database.
In this case, use the following output template:

{{
"action": "SEARCH",
"reasoning": "<add your reasoning here>"
}}

If you can answer the QUESTION using CONTEXT, use this template:

{{
"action": "ANSWER",
"answer": "<your answer>",
"source": "CONTEXT"
}}

If the context doesn't contain the answer, use your own knowledge to answer the question

{{
"action": "ANSWER",
"answer": "<your answer>",
"source": "OWN_KNOWLEDGE"
}}
""".strip()

In [27]:
question = "How can I run Docker on Windows 10?"
context = 'EMPTY'

In [28]:
prompt = prompt_template.format(question=question,  context=context)

In [29]:
print(prompt)

You're a course teaching assistant.

You're given a QUESTION from a course student and that you need to answer with your own knowledge and provided CONTEXT.
At the beginning the context is EMPTY.

<QUESTION>
How can I run Docker on Windows 10?
</QUESTION>

<CONTEXT> 
EMPTY
</CONTEXT>

If CONTEXT is EMPTY, you can use our FAQ database.
In this case, use the following output template:

{
"action": "SEARCH",
"reasoning": "<add your reasoning here>"
}

If you can answer the QUESTION using CONTEXT, use this template:

{
"action": "ANSWER",
"answer": "<your answer>",
"source": "CONTEXT"
}

If the context doesn't contain the answer, use your own knowledge to answer the question

{
"action": "ANSWER",
"answer": "<your answer>",
"source": "OWN_KNOWLEDGE"
}


In [30]:
answer = llm(prompt)

In [31]:
print(answer)

{
"action": "ANSWER",
"answer": "To run Docker on Windows 10, you need to follow these steps: 1. Ensure your Windows 10 version is 64-bit and supports Hyper-V (Windows 10 Pro, Enterprise, or Education). 2. Enable the Hyper-V feature by going to Control Panel > Programs > Turn Windows features on or off. Check 'Hyper-V' and click OK. 3. Download and install Docker Desktop for Windows from the official Docker website. 4. Once installed, launch Docker Desktop and follow any setup prompts. 5. You may need to log in to Docker Hub or create an account. 6. After setup, you can run Docker commands through PowerShell or Windows Command Prompt. Make sure to configure any necessary settings via the Docker Desktop application, such as allocating resources and configuring integration with WSL 2 if desired.",
"source": "OWN_KNOWLEDGE"
}


In [32]:
question = "Can I still join the course?"
context = 'EMPTY'

In [33]:
prompt = prompt_template.format(question=question,  context=context)

In [36]:
answer_json = llm(prompt)

In [37]:
import json

In [38]:
answer = json.loads(answer_json)

In [39]:
print(answer)

{'action': 'SEARCH', 'reasoning': "The question about joining the course doesn't have enough context provided to offer a direct answer, so I'll search our FAQ database for related information regarding enrollment."}


In [40]:
answer['action']

'SEARCH'

In [41]:
def build_context(search_results):
    context = ""

    for doc in search_results:
        context = context + f"section: {doc['section']}\nquestion: {doc['question']}\nanswer: {doc['text']}\n\n"
     
    return context.strip()

In [42]:
search_results = search(question)
context = build_context(search_results)
prompt = prompt_template.format(question=question, context=context)
print(prompt)

You're a course teaching assistant.

You're given a QUESTION from a course student and that you need to answer with your own knowledge and provided CONTEXT.
At the beginning the context is EMPTY.

<QUESTION>
Can I still join the course?
</QUESTION>

<CONTEXT> 
section: General course-related questions
question: Course - Can I still join the course after the start date?
answer: Yes, even if you don't register, you're still eligible to submit the homeworks.
Be aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute.

section: General course-related questions
question: Certificate - Can I follow the course in a self-paced mode and get a certificate?
answer: No, you can only get a certificate if you finish the course with a “live” cohort. We don't award certificates for the self-paced mode. The reason is you need to peer-review capstone(s) after submitting a project. You can only peer-review projects at the time the cour

In [43]:
answer_json = llm(prompt)

In [44]:
print(answer_json)

{
"action": "ANSWER",
"answer": "Yes, you can still join the course even after the start date. You are eligible to submit homework assignments, but keep in mind that there will be deadlines for turning in final projects. It's important not to leave everything to the last minute.",
"source": "CONTEXT"
}


In [45]:
def agentic_rag_v1(question):
    context = "EMPTY"
    prompt = prompt_template.format(question=question, context=context)
    answer_json = llm(prompt)
    answer = json.loads(answer_json)
    print(answer)

    if answer['action'] == 'SEARCH':
        print('need to perform search...')
        search_results = search(question)
        context = build_context(search_results)
        
        prompt = prompt_template.format(question=question, context=context)
        answer_json = llm(prompt)
        answer = json.loads(answer_json)
        print(answer)

    return answer

## "Agentic" Search

In [46]:
question = "How do i do well on module 1?"

In [62]:
def dedup(seq):
    seen = set()
    result = []
    for el in seq:
        _id = el['_id']
        if _id in seen:
            continue
        seen.add(_id)
        result.append(el)
    return result

search_results = dedup(search_results)

In [47]:
prompt_template = """
You're a course teaching assistant.

You're given a QUESTION from a course student and that you need to answer with your own knowledge and provided CONTEXT.

The CONTEXT is build with the documents from our FAQ database.
SEARCH_QUERIES contains the queries that were used to retrieve the documents
from FAQ to and add them to the context.
PREVIOUS_ACTIONS contains the actions you already performed.

At the beginning the CONTEXT is empty.

You can perform the following actions:

- Search in the FAQ database to get more data for the CONTEXT
- Answer the question using the CONTEXT
- Answer the question using your own knowledge

For the SEARCH action, build search requests based on the CONTEXT and the QUESTION.
Carefully analyze the CONTEXT and generate the requests to deeply explore the topic. 

Don't use search queries used at the previous iterations.

Don't repeat previously performed actions.

Don't perform more than {max_iterations} iterations for a given student question.
The current iteration number: {iteration_number}. If we exceed the allowed number 
of iterations, give the best possible answer with the provided information.

Output templates:

If you want to perform search, use this template:

{{
"action": "SEARCH",
"reasoning": "<add your reasoning here>",
"keywords": ["search query 1", "search query 2", ...]
}}

If you can answer the QUESTION using CONTEXT, use this template:

{{
"action": "ANSWER_CONTEXT",
"answer": "<your answer>",
"source": "CONTEXT"
}}

If the context doesn't contain the answer, use your own knowledge to answer the question

{{
"action": "ANSWER",
"answer": "<your answer>",
"source": "OWN_KNOWLEDGE"
}}

<QUESTION>
{question}
</QUESTION>

<SEARCH_QUERIES>
{search_queries}
</SEARCH_QUERIES>

<CONTEXT> 
{context}
</CONTEXT>

<PREVIOUS_ACTIONS>
{previous_actions}
</PREVIOUS_ACTIONS>
""".strip()

In [50]:
question = "How do i do well on module 1?"
max_iterations = 3
iteration_number = 0
search_queries = []
search_results = []
previous_actions = []

In [51]:
context = build_context(search_results)

prompt = prompt_template.format(
    question=question,
    context=context,
    search_queries="\n".join(search_queries),
    previous_actions='\n'.join([json.dumps(a) for a in previous_actions]),
    max_iterations=3,
    iteration_number=iteration_number
)
print(prompt)

You're a course teaching assistant.

You're given a QUESTION from a course student and that you need to answer with your own knowledge and provided CONTEXT.

The CONTEXT is build with the documents from our FAQ database.
SEARCH_QUERIES contains the queries that were used to retrieve the documents
from FAQ to and add them to the context.
PREVIOUS_ACTIONS contains the actions you already performed.

At the beginning the CONTEXT is empty.

You can perform the following actions:

- Search in the FAQ database to get more data for the CONTEXT
- Answer the question using the CONTEXT
- Answer the question using your own knowledge

For the SEARCH action, build search requests based on the CONTEXT and the QUESTION.
Carefully analyze the CONTEXT and generate the requests to deeply explore the topic. 

Don't use search queries used at the previous iterations.

Don't repeat previously performed actions.

Don't perform more than 3 iterations for a given student question.
The current iteration number

In [52]:
answer_json = llm(prompt)

In [53]:
print(answer_json)

{
"action": "SEARCH",
"reasoning": "The question is about doing well on module 1, and I need to gather information that might provide tips or guidelines specific to the success criteria for this module.",
"keywords": ["how to succeed in module 1", "tips for module 1", "module 1 best practices"]
}


In [72]:
answer = json.loads(answer_json)

In [73]:
keywords = answer['keywords']

In [58]:
print(keywords)

['how to succeed in module 1', 'tips for module 1', 'module 1 best practices']


In [59]:
for kw in keywords:
    search_queries.append(kw)
    sr = search (kw) 
    search_results.extend(sr) 

In [63]:
search_results = dedup(search_results)

In [64]:
len(search_results)

6

In [67]:
previous_actions.append(answer)

In [68]:
previous_actions

[{'action': 'SEARCH',
  'reasoning': 'The question is about doing well on module 1, and I need to gather information that might provide tips or guidelines specific to the success criteria for this module.',
  'keywords': ['how to succeed in module 1',
   'tips for module 1',
   'module 1 best practices']}]

In [69]:
iteration_number = 1
context = build_context(search_results)

prompt = prompt_template.format(
    question=question,
    context=context,
    search_queries="\n".join(search_queries),
    previous_actions='\n'.join([json.dumps(a) for a in previous_actions]),
    max_iterations=3,
    iteration_number=iteration_number
)
print(prompt)

You're a course teaching assistant.

You're given a QUESTION from a course student and that you need to answer with your own knowledge and provided CONTEXT.

The CONTEXT is build with the documents from our FAQ database.
SEARCH_QUERIES contains the queries that were used to retrieve the documents
from FAQ to and add them to the context.
PREVIOUS_ACTIONS contains the actions you already performed.

At the beginning the CONTEXT is empty.

You can perform the following actions:

- Search in the FAQ database to get more data for the CONTEXT
- Answer the question using the CONTEXT
- Answer the question using your own knowledge

For the SEARCH action, build search requests based on the CONTEXT and the QUESTION.
Carefully analyze the CONTEXT and generate the requests to deeply explore the topic. 

Don't use search queries used at the previous iterations.

Don't repeat previously performed actions.

Don't perform more than 3 iterations for a given student question.
The current iteration number

In [70]:
answer_json = llm(prompt)

In [71]:
print(answer_json)

{
"action": "SEARCH",
"reasoning": "I need to gather more specific strategies or tips related to Module 1: Docker and Terraform, as the context currently has limited information about success in this particular module.",
"keywords": ["Module 1 study tips", "succeed in Docker Terraform module", "best practices for Docker and Terraform"]
}


In [74]:
answer = json.loads(answer_json)
keywords = answer['keywords']
previous_actions.append(answer)
for kw in keywords:
    search_queries.append(kw)
    sr = search (kw) 
    search_results.extend(sr) 
search_results = dedup(search_results)

In [75]:
iteration_number = 2
context = build_context(search_results)

prompt = prompt_template.format(
    question=question,
    context=context,
    search_queries="\n".join(search_queries),
    previous_actions='\n'.join([json.dumps(a) for a in previous_actions]),
    max_iterations=3,
    iteration_number=iteration_number
)
print(prompt)

You're a course teaching assistant.

You're given a QUESTION from a course student and that you need to answer with your own knowledge and provided CONTEXT.

The CONTEXT is build with the documents from our FAQ database.
SEARCH_QUERIES contains the queries that were used to retrieve the documents
from FAQ to and add them to the context.
PREVIOUS_ACTIONS contains the actions you already performed.

At the beginning the CONTEXT is empty.

You can perform the following actions:

- Search in the FAQ database to get more data for the CONTEXT
- Answer the question using the CONTEXT
- Answer the question using your own knowledge

For the SEARCH action, build search requests based on the CONTEXT and the QUESTION.
Carefully analyze the CONTEXT and generate the requests to deeply explore the topic. 

Don't use search queries used at the previous iterations.

Don't repeat previously performed actions.

Don't perform more than 3 iterations for a given student question.
The current iteration number

In [76]:
answer_json = llm(prompt)
print(answer_json)

{
"action": "ANSWER",
"answer": "To do well in Module 1 (Docker and Terraform), consider the following tips:\n\n1. **Understand the Fundamentals**: Make sure you have a solid grasp of Docker containers and Terraform's infrastructure-as-code principles. Review documentation and beginner resources if needed.\n\n2. **Hands-On Practice**: The best way to learn Docker and Terraform is through practical experience. Build sample projects, try deploying applications, and experiment with different Terraform configurations.\n\n3. **Follow Best Practices**: Utilize best practices for Docker and Terraform. This includes organizing your Dockerfiles efficiently, using multi-stage builds in Docker, and structuring your Terraform code with modules for reusability.\n\n4. **Work on Real Projects**: Engage with projects that mimic real-world scenarios. This will help you understand how to apply what you're learning effectively.\n\n5. **Seek Help and Collaborate**: Don't hesitate to reach out to classmate

In [78]:
answer= json.loads(answer_json)
print(answer['answer'])

To do well in Module 1 (Docker and Terraform), consider the following tips:

1. **Understand the Fundamentals**: Make sure you have a solid grasp of Docker containers and Terraform's infrastructure-as-code principles. Review documentation and beginner resources if needed.

2. **Hands-On Practice**: The best way to learn Docker and Terraform is through practical experience. Build sample projects, try deploying applications, and experiment with different Terraform configurations.

3. **Follow Best Practices**: Utilize best practices for Docker and Terraform. This includes organizing your Dockerfiles efficiently, using multi-stage builds in Docker, and structuring your Terraform code with modules for reusability.

4. **Work on Real Projects**: Engage with projects that mimic real-world scenarios. This will help you understand how to apply what you're learning effectively.

5. **Seek Help and Collaborate**: Don't hesitate to reach out to classmates or forums for help when you encounter dif

In [79]:
question = "what do I need to do to be successful at module 1?"

search_queries = []
search_results = []
previous_actions = []


iteration = 0

while True:
    print(f'ITERATION #{iteration}...')

    context = build_context(search_results)
    prompt = prompt_template.format(
        question=question,
        context=context,
        search_queries="\n".join(search_queries),
        previous_actions='\n'.join([json.dumps(a) for a in previous_actions]),
        max_iterations=3,
        iteration_number=iteration
    )

    print(prompt)

    answer_json = llm(prompt)
    answer = json.loads(answer_json)
    print(json.dumps(answer, indent=2))

    previous_actions.append(answer)

    action = answer['action']
    if action != 'SEARCH':
        break

    keywords = answer['keywords']
    search_queries = list(set(search_queries) | set(keywords))
    
    for k in keywords:
        res = search(k)
        search_results.extend(res)

    search_results = dedup(search_results)
    
    iteration = iteration + 1
    if iteration >= 4:
        break

    print()

ITERATION #0...
You're a course teaching assistant.

You're given a QUESTION from a course student and that you need to answer with your own knowledge and provided CONTEXT.

The CONTEXT is build with the documents from our FAQ database.
SEARCH_QUERIES contains the queries that were used to retrieve the documents
from FAQ to and add them to the context.
PREVIOUS_ACTIONS contains the actions you already performed.

At the beginning the CONTEXT is empty.

You can perform the following actions:

- Search in the FAQ database to get more data for the CONTEXT
- Answer the question using the CONTEXT
- Answer the question using your own knowledge

For the SEARCH action, build search requests based on the CONTEXT and the QUESTION.
Carefully analyze the CONTEXT and generate the requests to deeply explore the topic. 

Don't use search queries used at the previous iterations.

Don't repeat previously performed actions.

Don't perform more than 3 iterations for a given student question.
The current 

In [80]:
answer

{'action': 'ANSWER',
 'answer': "To be successful in Module 1, focusing on the practical application of Docker and Terraform is key. Here are some general tips: \n1. **Hands-On Practice**: Since Module 1 involves tools like Docker and Terraform, practice by setting up your own projects using these technologies. This practical experience is invaluable.\n2. **Understand the Concepts**: Make sure you understand the basic concepts behind containerization and infrastructure as code. This will aid your ability to troubleshoot issues as you progress.\n3. **Join Study Groups**: Engaging with peers can help reinforce your learning. Consider forming study groups where you can discuss concepts and share insights.\n4. **Utilize Resources**: Access any supplementary materials provided in the course, such as tutorials or documentation on Docker and Terraform.\n5. **Ask for Help**: If you're encountering specific problems (like the module-not-found errors), don’t hesitate to reach out to your instruc

In [81]:
iteration

2

## Function Calling ("Tools Use")

In [82]:
def search(query):
    boost = {'question': 3.0, 'section': 0.5}

    results = index.search(
        query=query,
        filter_dict={'course': 'data-engineering-zoomcamp'},
        boost_dict=boost,
        num_results=5,
        output_ids=True
    )

    return results

In [83]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [113]:
question = "How do I do well in module 1?"

developer_prompt = """
You're a course teaching assistant. 
You're given a question from a course student and your task is to answer it.
""".strip()

tools = [search_tool]

chat_messages = [
    {"role": "developer", "content": developer_prompt},
    {"role": "user", "content": question}
]

response = client.responses.create(
    model='gpt-4o-mini',
    input=chat_messages,
    tools=tools
)
response.output

[ResponseFunctionToolCall(arguments='{"query":"module 1 tips"}', call_id='call_tDhZGLCarqFcLwaM0uE7FJ3l', name='search', type='function_call', id='fc_687e45721e94819e8d5654bc4729010b0fe997831c580f18', status='completed')]

In [114]:
calls = response.output

In [115]:
call = calls[0]

In [116]:
f_name = call.name

In [117]:
arguments = json.loads(call.arguments)

In [118]:
arguments

{'query': 'module 1 tips'}

In [119]:
f = globals()[f_name]

In [120]:
search_results = f(**arguments)

In [121]:
chat_messages.append(call)

chat_messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": json.dumps(search_results),
})

In [122]:
chat_messages

[{'role': 'developer',
  'content': "You're a course teaching assistant. \nYou're given a question from a course student and your task is to answer it."},
 {'role': 'user', 'content': 'How do I do well in module 1?'},
 ResponseFunctionToolCall(arguments='{"query":"module 1 tips"}', call_id='call_tDhZGLCarqFcLwaM0uE7FJ3l', name='search', type='function_call', id='fc_687e45721e94819e8d5654bc4729010b0fe997831c580f18', status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_tDhZGLCarqFcLwaM0uE7FJ3l',
  'output': '[{"text": "Following dbt with BigQuery on Docker readme.md, after `docker-compose build` and `docker-compose run dbt-bq-dtc init`, encountered error `ModuleNotFoundError: No module named \'pytz\'`\\nSolution:\\nAdd `RUN python -m pip install --no-cache pytz` in the Dockerfile under `FROM --platform=$build_for python:3.9.9-slim-bullseye as base`", "section": "Module 4: analytics engineering with dbt", "question": "DBT - Error: No module named \'pytz\' while settin

In [123]:
response = client.responses.create(
    model='gpt-4o-mini',
    input=chat_messages,
    tools=tools
)
response.output

[ResponseOutputMessage(id='msg_687e4581e4d8819e82a7d7a6a4f65e140fe997831c580f18', content=[ResponseOutputText(annotations=[], text="To do well in Module 1, here are some tips:\n\n1. **Understand the Basics**: Focus on grasping the foundational concepts of Docker and Terraform. These are essential tools often used in data engineering. Make sure you review any introductory materials provided.\n\n2. **Hands-On Practice**: Engage with hands-on exercises. Practical experience will solidify your understanding and help you troubleshoot common issues.\n\n3. **Follow Instructions Carefully**: When setting up your environment, closely follow any setup instructions, particularly for Docker. Common issues stem from misconfigurations.\n\n4. **Resolve Common Errors**: Familiarize yourself with common errors related to module imports (e.g., `ModuleNotFoundError`) and learn how to resolve them, such as:\n   - Ensuring all required libraries (like `psycopg2`) are installed properly.\n   - Updating pack

In [125]:
print(response.output[0].content[0].text)

To do well in Module 1, here are some tips:

1. **Understand the Basics**: Focus on grasping the foundational concepts of Docker and Terraform. These are essential tools often used in data engineering. Make sure you review any introductory materials provided.

2. **Hands-On Practice**: Engage with hands-on exercises. Practical experience will solidify your understanding and help you troubleshoot common issues.

3. **Follow Instructions Carefully**: When setting up your environment, closely follow any setup instructions, particularly for Docker. Common issues stem from misconfigurations.

4. **Resolve Common Errors**: Familiarize yourself with common errors related to module imports (e.g., `ModuleNotFoundError`) and learn how to resolve them, such as:
   - Ensuring all required libraries (like `psycopg2`) are installed properly.
   - Updating packages if you encounter issues.

5. **Use the Community**: Don’t hesitate to leverage course forums or study groups if you find yourself stuck. 

### Multiple calls

In [127]:
question = "How do I do well in module 1?"

developer_prompt = """
You're a course teaching assistant. 
You're given a question from a course student and your task is to answer it.
If you look up something in FAQ, convert the student question into multiple queries.
""".strip()

tools = [search_tool]

chat_messages = [
    {"role": "developer", "content": developer_prompt},
    {"role": "user", "content": question}
]

response = client.responses.create(
    model='gpt-4o-mini',
    input=chat_messages,
    tools=tools
)
response.output

[ResponseFunctionToolCall(arguments='{"query":"tips for doing well in module 1"}', call_id='call_XO0f5kxT8RXmWFiES9MW30d6', name='search', type='function_call', id='fc_687e464c1d1881a186b9d70047ef82d90bb1b45c34178b72', status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"how to study for module 1"}', call_id='call_ACsatMwxOO2CaJlxFd6ewdoy', name='search', type='function_call', id='fc_687e464c813081a1a72ec2dc8a1680b10bb1b45c34178b72', status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"resources for module 1 success"}', call_id='call_B3UaP3BV8Tgq6MEJ86pqBKYi', name='search', type='function_call', id='fc_687e464ce23481a180468e02e954b83d0bb1b45c34178b72', status='completed')]

In [129]:
calls = response.output

In [135]:
for call in calls:
    f_name = call.name
    arguments = json.loads(call.arguments)
    f = globals()[f_name]
    results = f(**arguments) 
    
    chat_messages.append(call)
    
    chat_messages.append({
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": json.dumps(results),
    })

In [136]:
response = client.responses.create(
    model='gpt-4o-mini',
    input=chat_messages,
    tools=tools
)
response.output

[ResponseOutputMessage(id='msg_687e47261a6081a19ef19120a882be900bb1b45c34178b72', content=[ResponseOutputText(annotations=[], text="To do well in Module 1 of the course, here are some key strategies and tips:\n\n1. **Understand Key Concepts**:\n   - Familiarize yourself with core topics such as Docker, Terraform, and SQLAlchemy.\n   - Review the installation processes and configurations necessary for the modules.\n\n2. **Practice Installing PostgreSQL**:\n   - Ensure you can install the PostgreSQL library. If you encounter issues, use the following commands:\n     - Install: `pip install psycopg2-binary`\n     - Upgrade if already installed: `pip install psycopg2-binary --upgrade`\n     - If you get a **ModuleNotFoundError**, consider running `conda update -n base -c defaults conda` or reinstall the package.\n\n3. **Utilize Jupyter Notebooks Effectively**:\n   - When using Jupyter notebooks, ensure you have all necessary packages installed to avoid module errors. This includes using `!

In [137]:
print(response.output[0].content[0].text)

To do well in Module 1 of the course, here are some key strategies and tips:

1. **Understand Key Concepts**:
   - Familiarize yourself with core topics such as Docker, Terraform, and SQLAlchemy.
   - Review the installation processes and configurations necessary for the modules.

2. **Practice Installing PostgreSQL**:
   - Ensure you can install the PostgreSQL library. If you encounter issues, use the following commands:
     - Install: `pip install psycopg2-binary`
     - Upgrade if already installed: `pip install psycopg2-binary --upgrade`
     - If you get a **ModuleNotFoundError**, consider running `conda update -n base -c defaults conda` or reinstall the package.

3. **Utilize Jupyter Notebooks Effectively**:
   - When using Jupyter notebooks, ensure you have all necessary packages installed to avoid module errors. This includes using `!pip install findspark` if working with Spark.

4. **Explore Course Resources**:
   - Access any provided documentation or resources related to Do

In [138]:
def do_call(tool_call_response):
    function_name = tool_call_response.name
    arguments = json.loads(tool_call_response.arguments)

    f = globals()[function_name]
    result = f(**arguments)

    return {
        "type": "function_call_output",
        "call_id": tool_call_response.call_id,
        "output": json.dumps(result, indent=2),
    }

In [140]:
response = client.responses.create(
    model='gpt-4o-mini',
    input=chat_messages,
    tools=tools
)
response.output

[ResponseOutputMessage(id='msg_687e487cb3d881a19ce5b2bc8e3a35d60bb1b45c34178b72', content=[ResponseOutputText(annotations=[], text="If you have specific questions or need clarification on certain topics, please let me know! I'm here to help you succeed in your course.", type='output_text', logprobs=[])], role='assistant', status='completed', type='message')]

In [141]:
for entry in response.output:
    chat_messages.append(entry)
    print(entry.type)

    if entry.type == 'function_call':      
        result = do_call(entry)
        chat_messages.append(result)
    elif entry.type == 'message':
        print(entry.text) 

message


AttributeError: 'ResponseOutputMessage' object has no attribute 'text'

In [142]:
developer_prompt = """
You're a course teaching assistant. 
You're given a question from a course student and your task is to answer it.

Use FAQ if your own knowledge is not sufficient to answer the question.
When using FAQ, perform deep topic exploration: make one request to FAQ,
and then based on the results, make more requests.

At the end of each response, ask the user a follow up question based on your answer.
""".strip()

chat_messages = [
    {"role": "developer", "content": developer_prompt},
]

In [143]:
while True: # main Q&A loop
    question = input() # How do I do my best for module 1?
    if question == 'stop':
        break

    message = {"role": "user", "content": question}
    chat_messages.append(message)

    while True: # request-response loop - query API till get a message
        response = client.responses.create(
            model='gpt-4o-mini',
            input=chat_messages,
            tools=tools
        )

        has_messages = False
        
        for entry in response.output:
            chat_messages.append(entry)
        
            if entry.type == 'function_call':      
                print('function_call:', entry)
                print()
                result = do_call(entry)
                chat_messages.append(result)
            elif entry.type == 'message':
                print(entry.content[0].text)
                print()
                has_messages = True

        if has_messages:
            break

 how do i do well in module 1


function_call: ResponseFunctionToolCall(arguments='{"query":"how to do well in module 1"}', call_id='call_x8h0t208mp2e90l5KFVdW5HI', name='search', type='function_call', id='fc_687e48f176c4819cb5b475ac94fb12b90af84f494a86a785', status='completed')

To excel in Module 1 of the course, which focuses on Docker and Terraform, here are some tips based on best practices and common issues:

1. **Understand Core Concepts**: Ensure you have a solid grasp of the basics of Docker and Terraform. Familiarize yourself with containerization concepts, Dockerfile syntax, and the Terraform configuration language.

2. **Hands-On Practice**: Implement what you've learned. Create your own Docker containers and practice writing Terraform scripts. Hands-on experience is crucial for learning these tools effectively.

3. **Troubleshooting**: Be prepared to encounter issues, especially regarding package installations. For example:
   - If you see errors like `ModuleNotFoundError: No module named 'psycopg2'`, en

 docker


function_call: ResponseFunctionToolCall(arguments='{"query":"how to do well in Docker module"}', call_id='call_f4dGcqgE4JjdicnOqcoELluS', name='search', type='function_call', id='fc_687e490906d0819cb823637f1a7c2fbd0af84f494a86a785', status='completed')

To excel in the Docker portion of Module 1, here are some actionable tips tailored to common practices and issues you might face:

1. **Understand Docker Fundamentals**:
   - Learn the concepts of images, containers, Dockerfile, and Docker Compose. Understanding how these components interact is crucial.

2. **Hands-On Practice**:
   - Create your own Docker images using Dockerfile. Experiment with different configurations and commands (`docker build`, `docker run`, etc.) to become comfortable with the command line.

3. **Use Docker Compose**:
   - Practice using Docker Compose to manage multi-container applications. Understand how to define services in `docker-compose.yml` and the relationships between them.

4. **Address Common Errors*

 stop


## Agent Multiple Tools

In [144]:
!wget https://raw.githubusercontent.com/alexeygrigorev/rag-agents-workshop/refs/heads/main/chat_assistant.py

--2025-07-21 22:08:05--  https://raw.githubusercontent.com/alexeygrigorev/rag-agents-workshop/refs/heads/main/chat_assistant.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8002::154, 2606:50c0:8000::154, 2606:50c0:8003::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8002::154|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3485 (3.4K) [text/plain]
Saving to: 'chat_assistant.py'

     0K ...                                                   100%  157K=0.02s

2025-07-21 22:08:05 (157 KB/s) - 'chat_assistant.py' saved [3485/3485]



In [159]:
def add_entry(question, answer):
    doc = {
        'question': question,
        'text': answer,
        'section': 'user added',
        'course': 'data-engineering-zoomcamp'
    }
    index.append(doc)

In [160]:
add_entry_description = {
    "type": "function",
    "name": "add_entry",
    "description": "Add an entry to the FAQ database",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The question to be added to the FAQ database",
            },
            "answer": {
                "type": "string",
                "description": "The answer to the question",
            }
        },
        "required": ["question", "answer"],
        "additionalProperties": False
    }
}

In [158]:
import chat_assistant

tools = chat_assistant.Tools()
tools.add_tool(search, search_tool)

tools.get_tools()


[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'Search query text to look up in the course FAQ.'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [163]:
tools.add_tool(add_entry, add_entry_description)

In [164]:
tools.get_tools()


[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'Search query text to look up in the course FAQ.'}},
   'required': ['query'],
   'additionalProperties': False}},
 {'type': 'function',
  'name': 'add_entry',
  'description': 'Add an entry to the FAQ database',
  'parameters': {'type': 'object',
   'properties': {'question': {'type': 'string',
     'description': 'The question to be added to the FAQ database'},
    'answer': {'type': 'string', 'description': 'The answer to the question'}},
   'required': ['question', 'answer'],
   'additionalProperties': False}}]

In [146]:

developer_prompt = """
You're a course teaching assistant. 
You're given a question from a course student and your task is to answer it.

Use FAQ if your own knowledge is not sufficient to answer the question.

At the end of each response, ask the user a follow up question based on your answer.
""".strip()

chat_interface = chat_assistant.ChatInterface()

chat = chat_assistant.ChatAssistant(
    tools=tools,
    developer_prompt=developer_prompt,
    chat_interface=chat_interface,
    client=client
)

In [ ]:
chat.run()

You: how do i do well in module one?


You: add this to the faq database
